In [ ]:
!pip -q install transformers sentencepiece accelerate

In [ ]:
import random
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google-t5/t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Loaded on:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded on: cpu


In [ ]:
text = "The cat sat on the mat."
print(text)

The cat sat on the mat.


In [ ]:
def perturb_sentence(text, mask_len=2):
    words = text.split()
    if len(words) <= 2:
        return text

    # Pick one random span to mask
    start = random.randint(0, max(0, len(words) - mask_len))
    end = min(len(words), start + mask_len)

    masked_words = words[:start] + ["<extra_id_0>"] + words[end:]
    masked_text = " ".join(masked_words)

    inputs = tokenizer(masked_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=True,
            num_beams=1,
            temperature=1.0,
            max_new_tokens=32,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # T5 usually returns something like:
    # <extra_id_0> filled text </s>
    match = re.search(r"<extra_id_0>\s*(.*?)\s*(<extra_id_\d+>|</s>|$)", decoded)
    fill = match.group(1).strip() if match else ""

    perturbed_words = words[:start] + fill.split() + words[end:]
    perturbed_text = " ".join(perturbed_words)

    return masked_text, decoded, perturbed_text

In [ ]:
masked_text, decoded_output, perturbed_text = perturb_sentence(text)

print("Masked text:   ", masked_text)
print("Model output:   ", decoded_output)
print("Perturbed text: ", perturbed_text)

Masked text:    The cat <extra_id_0> the mat.
Model output:    <pad><extra_id_0> falls asleep on<extra_id_1> gets off<extra_id_2> crawls all over<extra_id_3> crawls over<extra_id_4> rests on<extra_id_5> scratches<extra_id_6> hit the mat.</s>
Perturbed text:  The cat falls asleep on the mat.


In [ ]:
!pip -q install transformers sentencepiece accelerate datasets

import random
import re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1) Load T5
model_name = "t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print("Loaded model on:", device)

# 2) Load HC3
from datasets import load_dataset

ds = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/Hello-SimpleAI/HC3/resolve/refs%2Fconvert%2Fparquet/all/train/0000.parquet"
    }
)

print(ds)


# 3) Pick one example sentence from HC3
row = ds["train"][0]
text = row["human_answers"][0]   # you can swap to chatgpt_answers[0] later if needed

print("\nOriginal text:\n", text)

# 4) Perturb function
def perturb_sentence(text, mask_len=2):
    words = text.split()
    if len(words) <= 2:
        return text, text, text

    start = random.randint(0, max(0, len(words) - mask_len))
    end = min(len(words), start + mask_len)

    masked_words = words[:start] + ["<extra_id_0>"] + words[end:]
    masked_text = " ".join(masked_words)

    inputs = tokenizer(masked_text, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=True,
            num_beams=1,
            temperature=1.0,
            max_new_tokens=32,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract what T5 filled in for <extra_id_0>
    match = re.search(r"<extra_id_0>\s*(.*?)\s*(<extra_id_\d+>|</s>|$)", decoded)
    fill = match.group(1).strip() if match else ""

    perturbed_words = words[:start] + fill.split() + words[end:]
    perturbed_text = " ".join(perturbed_words)

    return masked_text, decoded, perturbed_text

# 5) Run it
masked_text, decoded_output, perturbed_text = perturb_sentence(text)

print("\nMasked text:\n", masked_text)
print("\nModel output:\n", decoded_output)
print("\nPerturbed text:\n", perturbed_text)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loaded model on: cpu


all/train/0000.parquet:   0%|          | 0.00/39.3M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source'],
        num_rows: 24322
    })
})

Original text:
 Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you 're still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n't stay there for long , you still achieved the status . Hence , # 1 best seller .

Masked text:
 Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best fi

In [ ]:
def perturb_sentence(text, mask_len=1, n_tries=5):
    words = text.split()

    if len(words) <= mask_len + 1:
        return text, text, text

    for _ in range(n_tries):
        start = random.randint(0, len(words) - mask_len)
        end = start + mask_len

        masked_words = words[:start] + ["<extra_id_0>"] + words[end:]
        masked_text = " ".join(masked_words)

        inputs = tokenizer(masked_text, return_tensors="pt", truncation=True).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                do_sample=False,
                num_beams=4,
                max_new_tokens=24,
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

        match = re.search(r"<extra_id_0>\s*(.*?)\s*(<extra_id_\d+>|</s>|$)", decoded)
        fill = match.group(1).strip() if match else ""

        if fill:
            perturbed_words = words[:start] + fill.split() + words[end:]
            perturbed_text = " ".join(perturbed_words)
            return masked_text, decoded, perturbed_text

    return masked_text, decoded, text

In [ ]:
text = row["human_answers"][0]

In [ ]:
masked_text, decoded_output, perturbed_text = perturb_sentence(text)

print("Original text:\n", text)
print("\nMasked text:\n", masked_text)
print("\nModel output:\n", decoded_output)
print("\nPerturbed text:\n", perturbed_text)

Original text:
 Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you 're still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n't stay there for long , you still achieved the status . Hence , # 1 best seller .

Masked text:
 Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you 're still an " oscar - winning " film . Same thing for best sellers . <extra_id_0> IIRC 

In [ ]:
samples = [ds["train"][i]["human_answers"][0] for i in range(5)]

for i, text in enumerate(samples):
    masked_text, decoded_output, perturbed_text = perturb_sentence(text)
    print(f"\n--- Example {i+1} ---")
    print("Original:", text[:200], "...")
    print("Perturbed:", perturbed_text[:200], "...")



--- Example 1 ---
Original: Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won ...
Perturbed: Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won ...

--- Example 2 ---
Original: salt is good for not dying in car crashes and car crashes are worse for cars then salt . Some places use other things , but salt is really cheap compared to most alternatives , although sand is pretty ...
Perturbed: salt is good for not dying in car crashes and car crashes are worse for cars then salt . Some places use other things , but salt is really good in comparison to most alternatives , although sand is pr ...

--- Example 3 ---
Original: The way it works is that old TV stations got a certain amount of bandwi

In [ ]:
def generate_perturbations(text, n=5):
    results = []
    for _ in range(n):
        _, _, perturbed = perturb_sentence(text)
        results.append(perturbed)
    return results

In [ ]:
text = ds["train"][1]["human_answers"][0]

perturbs = generate_perturbations(text, n=5)

print("Original:\n", text[:200], "\n")

for i, p in enumerate(perturbs):
    print(f"Perturb {i+1}:\n", p[:200], "\n")

Original:
 salt is good for not dying in car crashes and car crashes are worse for cars then salt . Some places use other things , but salt is really cheap compared to most alternatives , although sand is pretty 

Perturb 1:
 salt is good for not dying in car crashes . Most places say car crashes are worse for cars then salt . Some places use other things , but salt is really cheap compared to most alternatives , although  

Perturb 2:
 salt is good for not dying in car crashes and car crashes are worse in Utah then salt . Some places use other things , but salt is really cheap compared to most alternatives , although sand is pretty  

Perturb 3:
 salt is good for not dying in car crashes and car crashes are worse for cars then salt . Some places use other things , but salt is really cheap compared to most alternatives , although sand is pretty 

Perturb 4:
 salt is good for not dying in car crashes , but car crashes are worse for cars then salt . Some places use other things , but sal